# Edit state — fix the draft instead of re-running

Approve/deny is too coarse when the agent's draft is *almost* right. `Command(update={...})` ships a state delta back into the graph alongside the resume value, letting a reviewer patch the LLM's output in place — one human action instead of an LLM round-trip.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / '04-human-in-the-loop'))
os.environ.setdefault('LLM_CACHE_ONLY', '1')

from hitl import build_research_graph, Checkpointer, Command, get_question

graph = build_research_graph(); cp = Checkpointer()
state = graph.run({'question': get_question('q01'), 'question_id': 'q01'},
                  thread_id='edit-demo', checkpointer=cp)
print('LLM draft:', state['draft'][:120])

In [ ]:
# Reviewer rewrites the draft and approves in one Command.
edited = '[synth-001] RA-MoE reduces p99 decode latency by 38% vs. standard learned routing.'
state = graph.resume(thread_id='edit-demo',
                     command=Command(
                         resume={'approved': True, 'reviewer': 'editor@team'},
                         update={'draft': edited}),
                     checkpointer=cp)
print('downstream draft:', state['draft'])
print('published     :', state['final_answer'])
assert state['final_answer'] == edited

## Why this works

* `Command.update` is merged into the resumed state **before** the interrupted node re-runs.
* The interrupted node sees the human's edit in `state['draft']` and proceeds as if the LLM had written it.
* The checkpointer's history shows two writes to `draft` — LLM, then human — so audits can attribute the change.

Compare with the approval-gates leaf: there, the human's only freedom was {approve, deny}. Here they can change the substance. Both patterns coexist; pick per tool.